In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import numpy as np
import string
import os
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

/Users/teo/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class CaptchaPreprocessor:
    """
    Preprocessor for segmenting and preparing CAPTCHA images.
    """
    def __init__(self, target_size=(32, 32)):
        self.target_size = target_size
    
    def segment_and_preprocess(self, image_path: str) -> torch.Tensor:
        """
        Segment CAPTCHA and prepare for model input.
        
        Args:
            image_path: Path to CAPTCHA image
        
        Returns:
            Tensor of shape (1, num_chars, 1, height, width)
        """
        # Read and segment characters
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Could not read image: {image_path}")
        
        processed_img = remove_black_lines(img)
        chars = segment_captcha_characters(processed_img)
        
        # Handle case where no characters were segmented
        if len(chars) == 0:
            print(f"⚠️ Warning: No characters segmented from {image_path}")
            # Create a dummy character (blank image) to avoid crash
            dummy_char = np.zeros(self.target_size, dtype=np.float32)
            dummy_char = dummy_char[np.newaxis, ...]  # Add channel dimension
            char_tensor = torch.from_numpy(dummy_char[np.newaxis, ...])  # (1, 1, 1, H, W)
            return char_tensor
        
        # Preprocess each character
        processed_chars = []
        for char_img in chars:
            # Resize to target size
            resized = cv2.resize(char_img, self.target_size, interpolation=cv2.INTER_AREA)
            
            # Normalize to [0, 1]
            normalized = resized.astype(np.float32) / 255.0
            
            # Add channel dimension
            normalized = normalized[np.newaxis, ...]  # (1, height, width)
            
            processed_chars.append(normalized)
        
        # Stack into tensor
        char_tensor = np.stack(processed_chars, axis=0)  # (num_chars, 1, height, width)
        char_tensor = torch.from_numpy(char_tensor).unsqueeze(0)  # (1, num_chars, 1, height, width)
        
        return char_tensor

In [2]:
# Define the set of characters you expect to see
CHARSET = string.digits + string.ascii_lowercase

# --- Build the Vocabulary ---
# Start with the special <UNK> token at index 0
CHAR_TO_IDX = {'<UNK>': 0}

# Add the rest of the characters from the charset, starting from index 1
for char in CHARSET:
    # This check prevents adding duplicate characters, if any
    if char not in CHAR_TO_IDX:
        CHAR_TO_IDX[char] = len(CHAR_TO_IDX)

# The reverse mapping from index to character
IDX_TO_CHAR = {idx: char for char, idx in CHAR_TO_IDX.items()}

# The total number of classes is the size of our vocabulary
NUM_CLASSES = len(CHAR_TO_IDX)

print(f"Number of classes (vocabulary size): {NUM_CLASSES}")

Number of classes (vocabulary size): 37


In [ ]:
class CaptchaDataset(Dataset):
    """
    Improved Dataset class for CAPTCHA images.

    - Image preprocessing and segmentation are done on-the-fly.
    - Padding is deferred to a custom `collate_fn` for batch-level efficiency.    
    """
    def __init__(self, image_paths, labels, char_map=CHAR_TO_IDX, target_size=(32, 32)):
        """
        Args:
            image_paths (list): List of paths to CAPTCHA images.
            labels (list): List of string labels corresponding to each image.
            char_map (dict): A dictionary mapping characters to integer indices.
            target_size (tuple): The size to resize each character image to.
        """
        self.image_paths = image_paths
        self.labels = labels
        self.target_size = target_size
        self.char_map = char_map
        self.preprocessor = CaptchaPreprocessor(target_size=self.target_size)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        """
        Retrieves a single pre-processed CAPTCHA and its label tensor.

        Returns:
            tuple: A tuple containing:
                - char_tensor (torch.Tensor): Tensor of shape (num_chars, 1, H, W).
                - label_indices (torch.Tensor): Tensor of shape (num_chars,) with character indices.
        """
        image_path = self.image_paths[idx]
        label = self.labels[idx]

        # Segment and preprocess the image, returns (1, num_chars, 1, H, W)
        char_tensor = self.preprocessor.segment_and_preprocess(image_path)
        # Remove the batch dimension -> (num_chars, 1, H, W)
        char_tensor = char_tensor.squeeze(0)

        # Convert label string to a tensor of character indices.
        # Use the index for '<UNK>' (assumed to be 0) for any character not in the map.
        label_indices = [self.char_map.get(char, 0) for char in label]
        label_tensor = torch.tensor(label_indices, dtype=torch.long)
        
        return char_tensor, label_tensor


def collate_fn_padding(batch):
    """
    Custom collate function to pad sequences in a batch to the same length.
    This is passed to the `DataLoader`.

    Args:
        batch (list): A list of tuples, where each tuple is (char_tensor, label_tensor).

    Returns:
        tuple: A tuple containing:
            - padded_images (torch.Tensor): Padded image tensors of shape (B, max_len, 1, H, W).
            - padded_labels (torch.Tensor): Padded label tensors of shape (B, max_len).
            - lengths (torch.Tensor): Original sequence lengths for each item.
    """
    # Unzip the batch of (image, label) tuples
    images, labels = zip(*batch)
    images = list(images)
    labels = list(labels)

    # Get original lengths of labels before padding
    lengths = torch.tensor([len(label) for label in labels], dtype=torch.long)

    # Pad image tensors. `pad_sequence` expects (seq_len, *), which matches our tensors.
    # We set `batch_first=True` to get the conventional (batch, seq_len, C, H, W) shape.
    padded_images = pad_sequence(images, batch_first=True, padding_value=0.0)

    # Pad label tensors. The `padding_value` should be the index that the loss function ignores.
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-1)

    return padded_images, padded_labels, lengths


def load_captcha_dataset(data_dir, train_ratio=0.8, expected_chars=6, 
                         target_size=(32, 32), random_seed=42):
    """
    Load CAPTCHA dataset from directory and split into train/validation sets.
    
    Args:
        data_dir: Directory containing CAPTCHA images (filename = label)
        train_ratio: Ratio of training data (default 0.8 = 80%)
        expected_chars: Number of characters in each CAPTCHA
        target_size: Size to resize each character to
        random_seed: Random seed for reproducible splits
    
    Returns:
        train_dataset: Training dataset
        val_dataset: Validation dataset
    """
    # Get all image files
    image_files = []
    labels = []
    skipped = 0
    
    preprocessor = CaptchaPreprocessor(target_size=target_size)
    
    for filename in os.listdir(data_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            # Extract label from filename (remove extension)
            label = os.path.splitext(filename)[0].split('-')[0]
            
            image_path = os.path.join(data_dir, filename)
            
            # Test if image can be segmented
            try:
                img = cv2.imread(image_path)
                if img is None:
                    skipped += 1
                    continue
                processed_img = remove_black_lines(img)
                chars = segment_captcha_characters(processed_img)
                
                # Skip if no characters detected or too few/many
                if len(chars) == 0:
                    print(f"Skipping {filename}: No characters detected")
                    skipped += 1
                    continue
                    
                image_files.append(image_path)
                labels.append(label.upper())  # Convert to uppercase
            except Exception as e:
                print(f"Skipping {filename}: {str(e)}")
                skipped += 1
                continue
    
    print(f"Found {len(image_files)} valid CAPTCHA images in {data_dir}")
    if skipped > 0:
        print(f"Skipped {skipped} images due to errors")
    
    if len(image_files) == 0:
        raise ValueError(f"No valid images found in {data_dir}")
    
    # Split into train and validation sets
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        image_files, labels, 
        train_size=train_ratio, 
        random_state=random_seed,
        shuffle=True
    )
    
    print(f"Training samples: {len(train_paths)}")
    print(f"Validation samples: {len(val_paths)}")
    
    # Create datasets
    train_dataset = CaptchaDataset(train_paths, train_labels, target_size=target_size)
    val_dataset = CaptchaDataset(val_paths, val_labels, target_size=target_size)
    
    return train_dataset, val_dataset


def create_dataloaders(data_dir, batch_size=32, train_ratio=0.8, expected_chars=6,
                       target_size=(32, 32), random_seed=42):
    """
    Create train and validation DataLoaders from CAPTCHA dataset.
    
    Args:
        data_dir: Directory containing CAPTCHA images
        batch_size: Batch size for training
        train_ratio: Ratio of training data (0.8 = 80%)
        expected_chars: Number of characters in each CAPTCHA
        target_size: Size to resize each character to
        num_workers: Number of worker processes for data loading
        random_seed: Random seed for reproducible splits
    
    Returns:
        train_loader: DataLoader for training
        val_loader: DataLoader for validation
    """
    # Load datasets
    train_dataset, val_dataset = load_captcha_dataset(
        data_dir, train_ratio, expected_chars, target_size, random_seed
    )

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn_padding,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn_padding,
    )
    
    return train_loader, val_loader

In [ ]:
class CharacterCNN(nn.Module):
    """
    CNN module for extracting features from individual character images.
    """
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.5):
        super(CharacterCNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(dropout)
        
        # Feature dimension for RNN
        self.feature_dim = 128
        
        # Classification head (used during training individual CNN)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x, return_features=False):
        """
        Args:
            x: Input tensor of shape (batch, 1, height, width)
            return_features: If True, return features for RNN instead of classification
        """
        # Input: (batch, 1, 32, 32)
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # (batch, 32, 16, 16)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # (batch, 64, 8, 8)
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # (batch, 128, 4, 4)
        
        # Flatten
        x = x.view(x.size(0), -1)  # (batch, 128*4*4)
        
        if return_features:
            # Return features for RNN (before classification)
            features = F.relu(self.fc1(x))  # (batch, 256)
            return features
        else:
            # Full classification (for training CNN alone)
            x = F.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.fc2(x)
            return x


class CaptchaCNNRNN(nn.Module):
    """
    Combined CNN-RNN model for CAPTCHA recognition.
    This version has an integrated CNN backbone, removing the need for a separate CharacterCNN class.
    """
    def __init__(self, num_classes, hidden_size=256, num_layers=2, 
                 dropout=0.3, use_lstm=True):
        super(CaptchaCNNRNN, self).__init__()
        
        self.num_classes = num_classes
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.use_lstm = use_lstm
        
        # --- Integrated CNN Backbone ---
        # Processes each character image to extract features.
        self.cnn_backbone = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32x32 -> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16x16 -> 8x8
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2)   # 8x8 -> 4x4
        )
        
        # This layer projects the flattened CNN features to the dimension
        # expected by the RNN.
        self.feature_dim = 256  # The desired dimension for the sequence features
        cnn_output_size = 128 * 4 * 4  # 2048
        self.cnn_fc = nn.Linear(cnn_output_size, self.feature_dim)
        
        # --- RNN for Sequential Modeling ---
        if use_lstm:
            self.rnn = nn.LSTM(
                input_size=self.feature_dim,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0,
                bidirectional=True
            )
        else:
            self.rnn = nn.RNN(
                input_size=self.feature_dim,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0,
                bidirectional=True
            )
        
        # Bidirectional RNN output size is twice the hidden size
        self.rnn_output_size = hidden_size * 2
        
        # --- Final Classifier ---
        # Combines RNN output with the projected CNN features for the final prediction.
        self.classifier = nn.Sequential(
            nn.Linear(self.rnn_output_size + self.feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, char_images):
        """
        Args:
            char_images (Tensor): Input tensor of shape (batch, num_chars, 1, height, width).
        
        Returns:
            predictions (Tensor): Output tensor of shape (batch, num_chars, num_classes).
        """
        batch_size, num_chars, _, height, width = char_images.size()
        
        # Step 1: Reshape to process all character images through the CNN at once.
        char_images_flat = char_images.view(batch_size * num_chars, 1, height, width)
        
        # Step 2: Pass through CNN backbone and feature projection layer.
        cnn_out = self.cnn_backbone(char_images_flat)
        cnn_flat = cnn_out.view(cnn_out.size(0), -1)
        cnn_features = self.cnn_fc(cnn_flat)
        
        # Step 3: Reshape back into a sequence for the RNN.
        rnn_input = cnn_features.view(batch_size, num_chars, self.feature_dim)
        
        # Step 4: Feed sequence through RNN.
        rnn_output, _ = self.rnn(rnn_input)
        
        # Step 5: Combine RNN output with original CNN features.
        combined = torch.cat([rnn_output, rnn_input], dim=2)
        
        # --- FIX: Apply classifier directly to the sequence ---
        # This is more robust than flattening and reshaping.
        # The Linear layers in the classifier will automatically operate on the
        # last dimension (the features) of the (batch, num_chars, features) tensor.
        predictions = self.classifier(combined)
        
        return predictions
    
    def predict(self, char_images):
        """
        Predict characters from segmented images.
        
        Args:
            char_images: Tensor of shape (batch, num_chars, 1, height, width)
        
        Returns:
            predicted_text: List of predicted strings
        """
        self.eval()
        with torch.no_grad():
            predictions = self.forward(char_images)
            # Get argmax for each position
            predicted_indices = torch.argmax(predictions, dim=2)  # (batch, num_chars)
            
            # Convert to characters
            predicted_text = []
            for batch_idx in range(predicted_indices.size(0)):
                # This requires IDX_TO_CHAR to be defined in the script's scope
                chars = [IDX_TO_CHAR.get(int(idx.item()), '?') for idx in predicted_indices[batch_idx]]
                predicted_text.append(''.join(chars))
            
            return predicted_text

Number of classes (vocabulary size): 63


In [164]:
def run_epoch(model, dataloader, criterion, optimizer, device, is_training):
    """
    Helper function to run a single training or validation epoch.
    Reduces code duplication and adds progress monitoring.
    """
    # Set model to training or evaluation mode
    model.train(is_training)

    total_loss = 0.0
    total_chars_correct = 0
    total_chars = 0
    total_sequences_correct = 0
    total_sequences = 0

    # Add a progress bar for better user experience
    desc = "Training" if is_training else "Validation"
    progress_bar = tqdm(dataloader, desc=desc, leave=False)

    # Main loop
    for char_images, labels, lengths in progress_bar:
        char_images = char_images.to(device)  # (B, max_len, C, H, W)
        labels = labels.to(device)            # (B, max_len)

        # Forward pass
        with torch.set_grad_enabled(is_training):
            outputs = model(char_images)  # (B, max_len, num_classes)
            
            # DEBUG: Check shapes
            if outputs.shape[:2] != labels.shape:
                print(f"\n⚠️ Shape mismatch detected!")
                print(f"  char_images shape: {char_images.shape}")
                print(f"  outputs shape: {outputs.shape}")
                print(f"  labels shape: {labels.shape}")
                # Trim outputs to match labels length
                min_len = min(outputs.size(1), labels.size(1))
                outputs = outputs[:, :min_len, :]
                labels = labels[:, :min_len]
                print(f"  Trimmed to: outputs {outputs.shape}, labels {labels.shape}")
            
            # Create mask for valid (non-padded) positions
            mask = (labels != -1)  # (B, max_len)
            
            # Flatten everything properly
            # outputs: (B, max_len, num_classes) -> (B*max_len, num_classes)
            outputs_flat = outputs.reshape(-1, outputs.size(-1))
            # labels: (B, max_len) -> (B*max_len,)
            labels_flat = labels.reshape(-1)
            # mask: (B, max_len) -> (B*max_len,)
            mask_flat = mask.reshape(-1)
            
            # Apply mask to select only valid positions
            valid_outputs = outputs_flat[mask_flat]  # (num_valid, num_classes)
            valid_labels = labels_flat[mask_flat]    # (num_valid,)
            
            # Compute loss only on valid positions
            loss = criterion(valid_outputs, valid_labels)

            # Backward pass and optimization
            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item()

        # --- Metrics Calculation ---
        # Get predicted indices by taking the argmax along the class dimension
        _, predicted = torch.max(outputs, 2)  # (B, max_len)
        
        # 1. Per-character accuracy (on non-padded characters)
        total_chars_correct += ((predicted == labels) & mask).sum().item()
        total_chars += mask.sum().item()
        
        # 2. Full sequence accuracy (more meaningful metric)
        # A sequence is correct only if all its non-padded characters are predicted correctly.
        correct_sequences_mask = ((predicted == labels) | ~mask).all(dim=1)
        total_sequences_correct += correct_sequences_mask.sum().item()
        total_sequences += labels.size(0)  # Total number of sequences in the batch

        # Update progress bar with live metrics
        char_acc = 100 * total_chars_correct / total_chars if total_chars > 0 else 0
        seq_acc = 100 * total_sequences_correct / total_sequences if total_sequences > 0 else 0
        progress_bar.set_postfix(loss=loss.item(), char_acc=f"{char_acc:.2f}%", seq_acc=f"{seq_acc:.2f}%")

    avg_loss = total_loss / len(dataloader)
    char_accuracy = 100 * total_chars_correct / total_chars if total_chars > 0 else 0
    sequence_accuracy = 100 * total_sequences_correct / total_sequences if total_sequences > 0 else 0
    
    return avg_loss, char_accuracy, sequence_accuracy


def train_model(model, train_loader, val_loader, num_epochs=50, lr=0.001, device='cuda', save_path='best_captcha_model.pth'):
    """
    Train the CAPTCHA model with improved logging, metrics, and best practices.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=-1) # Ignore padding tokens
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # Scheduler adjusts LR based on validation accuracy
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5, verbose=True)

    best_val_seq_acc = 0.0
    
    print("Starting model training...")
    for epoch in range(num_epochs):
        print(f'\n--- Epoch [{epoch+1}/{num_epochs}] ---')
        
        # Training phase
        train_loss, train_char_acc, train_seq_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, is_training=True
        )
        print(f'Train      -> Loss: {train_loss:.4f}, Char Acc: {train_char_acc:.2f}%, Sequence Acc: {train_seq_acc:.2f}%')
        
        # Validation phase
        val_loss, val_char_acc, val_seq_acc = run_epoch(
            model, val_loader, criterion, None, device, is_training=False
        )
        print(f'Validation -> Loss: {val_loss:.4f}, Char Acc: {val_char_acc:.2f}%, Sequence Acc: {val_seq_acc:.2f}%')
        
        # Adjust learning rate based on the validation sequence accuracy
        scheduler.step(val_seq_acc)
        
        # Save the model if it has the best validation sequence accuracy so far
        if val_seq_acc > best_val_seq_acc:
            best_val_seq_acc = val_seq_acc
            torch.save(model.state_dict(), save_path)
            print(f'✅ New best model saved to {save_path} (Validation Sequence Acc: {val_seq_acc:.2f}%)')

    print("\nTraining finished.")
    print(f"🏆 Best validation sequence accuracy: {best_val_seq_acc:.2f}%")


# Inference function
def predict_captcha(model, image_path, device='cuda'):
    """
    Predict CAPTCHA text from image.
    
    Args:
        model: Trained CaptchaCNNRNN model
        image_path: Path to CAPTCHA image
        expected_chars: Expected number of characters
        device: 'cuda' or 'cpu'
    
    Returns:
        Predicted CAPTCHA text
    """
    model = model.to(device)
    model.eval()
    
    preprocessor = CaptchaPreprocessor()
    char_images = preprocessor.segment_and_preprocess(image_path)
    char_images = char_images.to(device)
    
    with torch.no_grad():
        predicted_text = model.predict(char_images)
    
    return predicted_text[0]

In [165]:
training_data_dir = './train'
train_loader, val_loader = create_dataloaders(
        data_dir=training_data_dir,
        batch_size=32,
        train_ratio=0.8,  # 80% train, 20% validation
        expected_chars=6,
        target_size=(32, 32),
        random_seed=42
    )

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = CaptchaCNNRNN(
    num_classes=NUM_CLASSES,
    hidden_size=256,
    num_layers=2,
    dropout=0.3,
    use_lstm=True
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
# # ==================== TRAINING ====================
# # Train the model
# train_model(
#     model=model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     num_epochs=50,
#     lr=0.001,
#     device=device
# )
    
# model.load_state_dict(torch.load('best_captcha_model.pth'))
# predicted_text = predict_captcha(model, './main/0abe-0.png', device=device)
# print(f"Predicted: {predicted_text}")


Found 8010 valid CAPTCHA images in ./train
Training samples: 6408
Validation samples: 1602
Training batches: 201
Validation batches: 51
Using device: cpu
Model parameters: 3,460,351
